# Notebook 2: Model Training and Evaluation

This notebook trains and compares multiple regression models to predict tree trunk count from polygon morphological features. It uses the feature data produced by Notebook 1.

**Models evaluated**: Ridge Regression, Random Forest, XGBoost, CatBoost (RMSE and Poisson loss)

**Evaluation**: 80/20 stratified train/test split, 5-fold cross-validation with hyperparameter tuning, diagnostic plots, and feature importance analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, learning_curve
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TARGET = 'Point_Coun'

## 2.1 Load Features and Select Feature Set

Choose which feature set to use. Based on our analysis, **the 5-feature set is recommended** (equivalent performance, simpler model).

In [ ]:
# --- CONFIGURATION: Choose feature set ---
USE_5_FEATURES = True  # Set to False to use the 20-feature set

if USE_5_FEATURES:
    data = pd.read_pickle("train_features_5.pkl")
    FEATURES = ['perimeter', 'area', 'compactness', 'perimeter_to_area', 'eccentricity']
    print(f"Using 5-feature set")
else:
    data = pd.read_pickle("train_features_20.pkl")
    FEATURES = [
        'area', 'perimeter', 'perimeter_to_area', 'compactness', 'convexity',
        'eccentricity', 'major_axis', 'minor_axis', 'aspect_ratio', 'mrr_area_ratio',
        'n_vertices', 'boundary_sinuosity', 'n_concavities', 'mean_radius',
        'radius_std', 'radius_cv', 'radius_ratio', 'equivalent_diameter',
        'convex_hull_deficit', 'l_ratio',
    ]
    print(f"Using 20-feature set")

data = data.dropna()
X = data[FEATURES]
y = data[TARGET]
print(f"Dataset: {len(data)} samples, {len(FEATURES)} features")
print(f"Target range: {y.min()} - {y.max()}, mean: {y.mean():.2f}, median: {y.median()}")

## 2.2 Train/Test Split

In [ ]:
# Stratified 80/20 split
bins = [0, 3, 5, 8, 15, 100]
y_binned = pd.cut(y, bins=bins, labels=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y_binned
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train target mean: {y_train.mean():.2f}, Test target mean: {y_test.mean():.2f}")

## 2.3 Model Training with Cross-Validation

Each model is tuned via 5-fold CV grid search on the training set. The test set is held out for final evaluation only.

In [ ]:
def evaluate(y_true, y_pred, label=""):
    """Compute all evaluation metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    rho, _ = spearmanr(y_true, y_pred)
    w1 = np.mean(np.abs(y_true - y_pred) <= 1) * 100
    w2 = np.mean(np.abs(y_true - y_pred) <= 2) * 100
    return {'Model': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
            'Spearman': rho, 'Within_1': w1, 'Within_2': w2}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Define models and hyperparameter grids
model_configs = {
    'Ridge': {
        'model': Ridge(),
        'params': {'alpha': [0.1, 1.0, 10.0, 100.0]},
        'n_jobs': -1,
    },
    'RandomForest': {
        'model': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        'params': {'n_estimators': [100, 300, 500], 'max_depth': [3, 5, 7]},
        'n_jobs': -1,
    },
    'XGBoost': {
        'model': xgb.XGBRegressor(random_state=RANDOM_STATE, device='cuda', verbosity=0),
        'params': {'n_estimators': [100, 300, 500], 'max_depth': [3, 5, 7],
                   'learning_rate': [0.01, 0.05, 0.1]},
        'n_jobs': -1,
    },
    'CatBoost_RMSE': {
        'model': CatBoostRegressor(task_type='GPU', verbose=0, random_seed=RANDOM_STATE,
                                    loss_function='RMSE'),
        'params': {'iterations': [100, 300, 500], 'depth': [3, 5, 7],
                   'learning_rate': [0.01, 0.05, 0.1]},
        'n_jobs': 1,
    },
    'CatBoost_Poisson': {
        'model': CatBoostRegressor(task_type='GPU', verbose=0, random_seed=RANDOM_STATE,
                                    loss_function='Poisson'),
        'params': {'iterations': [100, 300, 500], 'depth': [3, 5, 7],
                   'learning_rate': [0.01, 0.05, 0.1]},
        'n_jobs': 1,
    },
}

# Train and evaluate all models
results = []
trained_models = {}

for name, cfg in model_configs.items():
    print(f"Training {name}...", end=" ", flush=True)
    grid = GridSearchCV(
        cfg['model'], cfg['params'], cv=cv,
        scoring='neg_mean_absolute_error', refit=True, n_jobs=cfg['n_jobs']
    )
    grid.fit(X_train, y_train)

    y_pred = np.clip(np.round(grid.predict(X_test)), 1, None)
    metrics = evaluate(y_test.values, y_pred, name)
    results.append(metrics)
    trained_models[name] = grid.best_estimator_

    print(f"MAE={metrics['MAE']:.2f}, R2={metrics['R2']:.3f}, params={grid.best_params_}")

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print("\n--- Model Comparison ---")
results_df

## 2.4 Diagnostic Plots

In [ ]:
# Select best model by R2
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]
y_pred_best = np.clip(np.round(best_model.predict(X_test)), 1, None)
residuals = y_test.values - y_pred_best

print(f"Best model: {best_name}")

# Predicted vs Actual
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(y_test, y_pred_best, alpha=0.5, s=30)
lims = [min(y_test.min(), y_pred_best.min()) - 1, max(y_test.max(), y_pred_best.max()) + 1]
ax.plot(lims, lims, 'r--', linewidth=2, label='1:1 line')
ax.set_xlabel('Actual Trunk Count')
ax.set_ylabel('Predicted Trunk Count')
ax.set_title(f'Predicted vs Actual ({best_name})')
ax.legend()
ax.set_aspect('equal')

# Residuals
ax = axes[1]
ax.scatter(y_pred_best, residuals, alpha=0.5, s=30)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Predicted')
ax.set_ylabel('Residual (Actual - Predicted)')
ax.set_title('Residual Plot')

# Residual distribution
ax = axes[2]
ax.hist(residuals, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--')
ax.set_xlabel('Residual')
ax.set_ylabel('Frequency')
ax.set_title(f'Residual Distribution (mean={residuals.mean():.2f})')

fig.tight_layout()
plt.show()

In [ ]:
# Error by trunk count range
error_df = pd.DataFrame({'actual': y_test.values, 'abs_error': np.abs(residuals)})
bin_labels = ['2-3', '4-6', '7-10', '11-20', '21+']
error_df['range'] = pd.cut(error_df['actual'], bins=[0, 3, 6, 10, 20, 100], labels=bin_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

error_df.boxplot(column='abs_error', by='range', ax=axes[0])
axes[0].set_xlabel('Actual Trunk Count Range')
axes[0].set_ylabel('Absolute Error')
axes[0].set_title('Error by Trunk Count Range')
plt.sca(axes[0])
plt.title('Error by Trunk Count Range')

# Feature importance
ax = axes[1]
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'get_feature_importance'):
    importances = best_model.get_feature_importance()
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_)
else:
    importances = np.zeros(len(FEATURES))

pd.Series(importances, index=FEATURES).sort_values().plot.barh(ax=ax)
ax.set_xlabel('Importance')
ax.set_title(f'Feature Importance ({best_name})')

fig.suptitle('')
fig.tight_layout()
plt.show()

# Print error breakdown
print("Error breakdown by trunk count range:")
for label in bin_labels:
    subset = error_df[error_df['range'] == label]
    if len(subset) > 0:
        print(f"  {label}: n={len(subset)}, mean_error={subset['abs_error'].mean():.2f}, "
              f"median_error={subset['abs_error'].median():.2f}")

## 2.5 Train Final Model on Full Dataset

After evaluation, retrain the best model on the **entire** dataset (no hold-out) to maximize performance for deployment on new data.

In [ ]:
# Retrain best model on ALL data
print(f"Retraining {best_name} on full dataset ({len(X)} samples)...")

# Get the best hyperparameters from CV and retrain
best_params = {}
if hasattr(best_model, 'get_params'):
    best_params = best_model.get_params()

final_model = type(best_model)(**best_params)
final_model.fit(X, y)

# Also train a CatBoost model on all data (recommended for <=8 range)
print("Training CatBoost RMSE on full dataset for <=8 operational use...")
cb_final = CatBoostRegressor(
    task_type='GPU', verbose=0, random_seed=RANDOM_STATE,
    loss_function='RMSE', depth=3, iterations=100, learning_rate=0.1
)
cb_final.fit(X, y)

print("Done. Both models ready for saving.")

## 2.6 Save Trained Models

In [ ]:
import json

# Save models
joblib.dump(final_model, "final_model_ridge.joblib")
print(f"Saved: final_model_ridge.joblib ({best_name}, trained on {len(X)} samples)")

joblib.dump(cb_final, "final_model_catboost.joblib")
print(f"Saved: final_model_catboost.joblib (CatBoost RMSE, trained on {len(X)} samples)")

# Save model config (feature list + metadata)
config = {
    'features': FEATURES,
    'n_features': len(FEATURES),
    'target': TARGET,
    'target_crs': 2039,
    'n_training_samples': len(X),
    'best_model_type': best_name,
    'test_r2': float(results_df.iloc[0]['R2']),
    'test_mae': float(results_df.iloc[0]['MAE']),
}
with open("model_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved: model_config.json")
print(f"\nModel config:")
config